In [2]:
import pandas as pd
from pathlib import Path

pasta = Path("../data/raw")

ficheiros = {
    "customer": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dados = {name: pd.read_csv(pasta / f) for name, f in ficheiros.items()}

for name, df in dados.items():
    print(f"{name}: {df.shape}")




customer: (99441, 5)
geolocation: (1000163, 5)
orders: (99441, 8)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [3]:
orders = dados["orders"]

lista = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]

for col in lista:
    orders[col] = pd.to_datetime(orders[col])

print(orders.dtypes) 
print("\n")

order_items = dados["order_items"]
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

print(order_items.dtypes)
print("\n")

order_reviews = dados["order_reviews"]
lista_review = ["review_creation_date", "review_answer_timestamp"]

for col in lista_review:
    order_reviews[col] = pd.to_datetime(order_reviews[col])


print(order_reviews.dtypes)





order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object


review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [4]:
for name, df in dados.items():                  #Orders, order_reviews and products have null values
    print(f"=== Valores Nulos em {name} ===")
    print(df.isna().sum())



=== Valores Nulos em customer ===
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
=== Valores Nulos em geolocation ===
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
=== Valores Nulos em orders ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
=== Valores Nulos em order_items ===
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
=== Valores Nu

In [5]:
no_delivered_date = orders[orders["order_delivered_customer_date"].isna()]
count = no_delivered_date["order_status"].value_counts()
print(len(no_delivered_date))

pd.DataFrame({"N": count , "%": count/len(no_delivered_date) *100})


2965


,N,%
order_status,,
shipped,1107,37.335582
canceled,619,20.876897
unavailable,609,20.539629
invoiced,314,10.590219
processing,301,10.151771
delivered,8,0.269815
created,5,0.168634
approved,2,0.067454


Of the 2,965 orders without a delivery date, 2,957 make sense: they have not been delivered yet (shipped, invoiced, processing, created, approved) or never will be (canceled, unavailable). The exception is 8 orders with order_status = "delivered". I will keep them because they contribute to revenue, and they are automatically excluded from delivery-time metrics since their delivery date is missing. One possible explanation is that the delivery was not recorded, or that the orders were lost on the way to the customer, but the data does not confirm this.


In [6]:
delivered_without_data = orders.loc[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isna()), ["order_status", "order_delivered_customer_date"]]
print(delivered_without_data)
print(f"Total: {len(delivered_without_data)} orders")

print("\n")

no_delivered_with_data = orders.loc[(orders["order_status"] != "delivered") & (~ orders["order_delivered_customer_date"].isna()), ["order_status", "order_delivered_customer_date"]]
print(no_delivered_with_data)
print(f"Total: {len(no_delivered_with_data)} orders")

      order_status order_delivered_customer_date
3002     delivered                           NaT
20618    delivered                           NaT
43834    delivered                           NaT
79263    delivered                           NaT
82868    delivered                           NaT
92643    delivered                           NaT
97647    delivered                           NaT
98038    delivered                           NaT
Total: 8 orders


      order_status order_delivered_customer_date
2921      canceled           2018-03-21 22:03:51
8791      canceled           2016-10-16 14:36:59
58266     canceled           2016-10-14 15:07:11
59332     canceled           2016-10-19 18:47:43
92636     canceled           2016-11-08 10:58:34
94399     canceled           2016-11-09 14:53:50
Total: 6 orders


I will keep the 6 canceled orders that have a delivery date, but all canceled orders will be excluded from the revenue calculation. These could be orders that were returned after delivery.

In [7]:
products = dados["products"]

cols = ["product_category_name", "product_name_lenght",
        "product_description_lenght", "product_photos_qty"]

print(products.isna().sum())
print("\n")
print(products[cols].isna().all(axis=1).sum())



product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


610


The 610 products with a null product_category_name also have nulls in name length, description length and photos quantity, so they are products with no metadata. I will fill the category with "unknown". The 2 products with null weight and dimensions will stay as NaN: filling them with 0 would create false data and pull the mean down.

In [8]:
order_approved_Nat = orders[orders["order_approved_at"].isna()]
order_approved_Nat["order_status"].value_counts()


order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

Null values in order_approved_at make sense for canceled orders (141), which were canceled before approval, and for created orders (5), which are still being created. However, the 14 delivered orders should have an approval date, because an order cannot be delivered without being approved. We can also conclude that, of the 625 canceled orders (619 + 6), only 141 were canceled before being approved.

In [9]:
order_delivered_carrier_without_date = orders[orders["order_delivered_carrier_date"].isna()]
count = order_delivered_carrier_without_date["order_status"].value_counts()
print(count)



order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64


All statuses make sense with a null order_delivered_carrier_date, except for delivered (2 orders): an order that was delivered must have been handed over to the carrier. We can also see that only 75 canceled orders (625 − 550) reached the carrier before being canceled.

In [31]:
order_review = dados["order_reviews"]

print(order_review.isna().sum())




review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


Only "review_comment_title" and "review_comment_message" have null values. This is expected, because leaving a comment is optional when submitting a review. The key column, review_score, has no nulls.

In [38]:
delivered = orders[orders["order_status"] == "delivered"]
col = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]

soma = delivered[col].isna().any(axis=1).sum()

print("\n")
print(f"Percentage of orders that have a delivered order status but with null values at col: {round(soma/len(delivered) * 100, 2)}")

null_sum = delivered[col].isna().sum(axis=1)
orders.loc[null_sum[null_sum > 1].index, ["order_id", "order_delivered_carrier_date", "order_delivered_customer_date","order_approved_at"]]

order_review[order_review["order_id"] == "2d858f451373b04fb5c984a1cc2defaf"]



Percentage of orders that have a delivered order status but with null values at col: 0.02


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
51996,4e755f114e50d33b9ac6a56e0d7d3ea9,2d858f451373b04fb5c984a1cc2defaf,5,NaN,NaN,2017-06-25,2017-06-27 01:49:04


In [ ]:
for name, df in dados.items():
    print(f"=== Valores duplicados da tabela: {name} ===")
    print(f"duplicados: {df.duplicated().sum()} total: {len(df)} percentagem: {round(df.duplicated().sum()/len(df) * 100, 2)}")




=== Valores duplicados da tabela: customer ===
duplicados: 0 total: 99441 percentagem: 0.0 
=== Valores duplicados da tabela: geolocation ===
duplicados: 261831 total: 1000163 percentagem: 26.18 
=== Valores duplicados da tabela: orders ===
duplicados: 0 total: 99441 percentagem: 0.0 
=== Valores duplicados da tabela: order_items ===
duplicados: 0 total: 112650 percentagem: 0.0 
=== Valores duplicados da tabela: order_payments ===
duplicados: 0 total: 103886 percentagem: 0.0 
=== Valores duplicados da tabela: order_reviews ===
duplicados: 0 total: 99224 percentagem: 0.0 
=== Valores duplicados da tabela: products ===
duplicados: 0 total: 32951 percentagem: 0.0 
=== Valores duplicados da tabela: sellers ===
duplicados: 0 total: 3095 percentagem: 0.0 
=== Valores duplicados da tabela: category_translation ===
duplicados: 0 total: 71 percentagem: 0.0 


Only the geolocation table has fully duplicate rows (26.18%). This is expected, since the table stores several coordinates per zip code prefix. All the other tables have no duplicate rows.

In [79]:
dict = {"customer": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],           
    "order_payments": ["order_id", "payment_sequential"],   
    "order_reviews": ["review_id"],                        
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],}

for name, cols in dict.items():
    df = dados[name]
    repetidas = df.duplicated(subset=cols).sum()
    print(f"{name} chave={cols}  repetidas: {repetidas}") 


customer chave=['customer_id']  repetidas: 0
orders chave=['order_id']  repetidas: 0
order_items chave=['order_id', 'order_item_id']  repetidas: 0
order_payments chave=['order_id', 'payment_sequential']  repetidas: 0
order_reviews chave=['review_id']  repetidas: 814
products chave=['product_id']  repetidas: 0
sellers chave=['seller_id']  repetidas: 0
category_translation chave=['product_category_name']  repetidas: 0


In [82]:
customers = dados["customer"]

n_lines = len(customers)                                  
n_ids = customers["customer_id"].nunique()               
n_people = customers["customer_unique_id"].nunique()    

print(f"linhas: {n_lines} | customer_id: {n_ids} | customer_unique_id: {n_people}")

purchase_by_customer = customers.groupby("customer_unique_id").size()
repetentes = (purchase_by_customer > 1).sum() 

print(f"pessoas com mais de uma compra: {repetentes} ({round(repetentes / n_people * 100, 2)}%)")

linhas: 99441 | customer_id: 99441 | customer_unique_id: 96096
pessoas com mais de uma compra: 2997 (3.12%)


In [ ]:
print(dados["order_reviews"].duplicated(subset=["order_id"]).sum())
print(dados["order_reviews"].duplicated(subset=["order_id", "review_id"]).sum())


551
0


98673

review_id alone is repeated in 814 rows and order_id alone is repeated in 551 rows, but the combination of review_id and order_id is unique, so it is the key of the order_reviews table.

In [118]:
distinct = order_review["order_id"].nunique()
print(f"distinct orders: {distinct}, reviews done: {len(order_review)}")  #c88b1d1b157a9999ce368f218a407141     df56136b8031ecd28e200bb18e6ddb2e 

n_review = order_review["order_id"].value_counts()
more_than1_review = (n_review > 1).sum()
print(f"orders with more than 1 review: {more_than1_review}")

order_review[order_review["order_id"] == "df56136b8031ecd28e200bb18e6ddb2e"]



distinct orders: 98673, reviews done: 99224
orders with more than 1 review: 547


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
2952,c444278834184f72b1484dfe47de7f97,df56136b8031ecd28e200bb18e6ddb2e,5,NaN,NaN,2017-02-08,2017-02-14 13:58:48
13982,72a1098d5b410ae50fbc0509d26daeb9,df56136b8031ecd28e200bb18e6ddb2e,5,NaN,NaN,2017-02-07,2017-02-10 10:46:09
62728,44f3e54834d23c5570c1d010824d4d59,df56136b8031ecd28e200bb18e6ddb2e,5,NaN,NaN,2017-02-09,2017-02-09 09:07:28


In [115]:
order_review[order_review["order_id"] == "c88b1d1b157a9999ce368f218a407141"]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-22,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21,2017-07-26 13:45:15


In [132]:
sorted_reviews = order_review.sort_values("review_answer_timestamp", kind="stable")
reviews_clean = sorted_reviews.drop_duplicates(subset= "order_id", keep = "last")

print(f"before: {order_review.shape} | after: {reviews_clean.shape}")           
print(f"unique order_id: {reviews_clean['order_id'].is_unique}")           
print(f"removed rows: {len(order_review) - len(reviews_clean)}")


before: (99224, 7) | after: (98673, 7)
unique order_id: True
removed rows: 551


For each order I kept only the review with the most recent review_answer_timestamp, because it reflects the customer’s final opinion and has time precision, so there are no ties. The result is one review per order (98,673 rows).

In [160]:
geolocation = dados["geolocation"]
rows = len(geolocation)
distinct_prefix = geolocation["geolocation_zip_code_prefix"].nunique()

print(f"n of rows: {rows} distinct_prefix: {distinct_prefix}")
count = geolocation.groupby("geolocation_zip_code_prefix").size()
mean_of_rows_prefix = round(count.mean(),2)
max_prefix = count.idxmax()

print(f"Media de linhas por prefixo: {mean_of_rows_prefix}")
print(f"Prefixo com mais linhas: {max_prefix}")

df_merge = pd.merge(geolocation, dados["customer"], left_on = "geolocation_zip_code_prefix", right_on  = "customer_zip_code_prefix", how= "inner")

df_merge.shape

geo_clean = geolocation.groupby("geolocation_zip_code_prefix").agg(lat_median = ("geolocation_lat", "median"), lng_median = ("geolocation_lng", "median")).reset_index()

print("\n")

print(f"before: {geolocation.shape} | after: {geo_clean.shape}")           
print(f"prefix unique: {geo_clean['geolocation_zip_code_prefix'].is_unique}") 

check = pd.merge(dados["customer"], geo_clean,
                 left_on="customer_zip_code_prefix",
                 right_on="geolocation_zip_code_prefix",
                 how="left")                                               

print(f"customers: {len(dados['customer'])} | after merge: {len(check)}")  
print(f"customers without coordinates: {check['lat_median'].isna().sum()}")

n of rows: 1000163 distinct_prefix: 19015
Media de linhas por prefixo: 52.6
Prefixo com mais linhas: 24220


before: (1000163, 5) | after: (19015, 3)
prefix unique: True
customers: 99441 | after merge: 99441
customers without coordinates: 278


The geolocation table has 1,000,163 rows for only 19,015 distinct zip code prefixes. Joining it directly with customers produced 15,083,455 rows, against 99,441 customers, because each customer was repeated once for every row of its prefix. This would inflate any revenue calculated after the join.
To avoid this, I grouped the table by geolocation_zip_code_prefix and kept the median latitude and longitude of each prefix. I used the median instead of the mean because it is not affected by outliers, such as a wrong coordinate far from the others. The result has one row per prefix (19,015 rows), and a left join with customers keeps exactly 99,441 rows.

In [ ]:
product = dados["products"]
translation = dados["category_translation"] #product_category_name

to_english = {"portateis_cozinha_e_preparadores_de_alimentos":"kitchen_appliances_and_food_prep", "pc_gamer":"pc_gamer" }

left_join = pd.merge(product, translation, on = "product_category_name", how = "left")

sum_of_null = left_join["product_category_name"].isna().sum()

null_category = left_join[left_join["product_category_name"].isna()]

print(f"Total of null: {sum_of_null}")
print("\n")

without_english = left_join[(~ left_join["product_category_name"].isna()) & left_join["product_category_name_english"].isna()]

without_english["product_category_name"].value_counts()

product_clean = left_join.copy()

product_clean["product_category_name_english"] = (product_clean["product_category_name_english"].fillna(product_clean["product_category_name"].map(to_english)).fillna("unknown"))

print(f"rows: {len(product)} -> {len(left_join)} -> {len(product_clean)}")                          
print(f"nulls in English: {product_clean['product_category_name_english'].isna().sum()}")          
print(f"'unknown' in English: {(product_clean['product_category_name_english'] == 'unknown').sum()}")   
print(f"nulls in Portuguese: {product_clean['product_category_name'].isna().sum()}")               


Total of null: 610


